# Panel pretrain job-4 — chart-native encoder (Colab A100)

Arm: `chart-native-encoder` (hypotheses.jsonl, registered 2026-10-23 **before** the
render farm started). Step (2): SimCLR pretrain on the ~260k panels from job-3,
**no labels**. The linear-probe judge (step 3) runs on the laptop afterwards.

**Pre-committed** (in code/panel_simclr_colab.py docstring): ResNet-18 random init,
projection head 512→512→128; RandomResizedCrop 224 + mild brightness/contrast + blur,
**no flips** (time axis is directional); NT-Xent τ=0.2; AdamW 3e-4, cosine, 20 epochs,
batch 512, bf16. Embedding = backbone 512-d.

**Resume-safe**: epoch checkpoints + embedding parts live in Drive. Re-run any time.
Needs job-3 finished (`panels_part_*.zip` in the Drive panel_farm folder).
Runtime: **A100**. ~2h. Last cell unassigns the runtime when everything is done.

In [ ]:
import os
DRIVE_PANELS = "/content/drive/MyDrive/brain_packs/panel_farm"
OUT_DIR = "/content/drive/MyDrive/brain_packs/panel_embed"
RAW = "https://raw.githubusercontent.com/vicjayjay/brain-packs/main/code/"
CODE_FILES = ["panel_simclr_colab.py"]

In [ ]:
import subprocess, sys, urllib.request
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "pandas", "pyarrow", "torchvision"], check=True)
from google.colab import drive  # type: ignore
drive.mount("/content/drive")   # one consent

for f in CODE_FILES:
    dst = Path("/content") / f
    if dst.exists():
        dst.unlink()                       # always fresh
    urllib.request.urlretrieve(RAW + f + "?v=1", dst)
sys.path.insert(0, "/content")

import torch
assert torch.cuda.is_available(), "needs a GPU runtime (A100)"
print(torch.cuda.get_device_name(0), "— pack + code ready")

In [ ]:
# Stage 1: SimCLR pretrain (resume-safe, ~2h on A100)
import importlib, sys
import panel_simclr_colab as sim
importlib.reload(sim)
sys.argv = ["simclr", "pretrain", "--drive-panels", DRIVE_PANELS, "--out", OUT_DIR]
sim.main()

In [ ]:
# Stage 2: embed all panels with the frozen backbone -> Drive parquet + sha256 manifest
import importlib, sys
import panel_simclr_colab as sim
importlib.reload(sim)
sys.argv = ["simclr", "embed", "--drive-panels", DRIVE_PANELS, "--out", OUT_DIR]
sim.main()

In [ ]:
# Runs only when pretrain+embed finished. Everything durable is in Drive;
# unassign so the runtime stops billing immediately.
from google.colab import runtime  # type: ignore
runtime.unassign()